# An honest estimate: data splits, cross-validation and leakage

This notebook covers the first block of Session 7. In Session 6 you split the data once into a training and a test set and reported one test score. That score answers the question "how well will the model do on new data?" only roughly: another split gives another number. Here we build the tools that make the answer reliable: a clear role for each part of the data, cross-validation that uses every row for validation once, splitters that respect groups and time, and a bootstrap interval that says how uncertain a score is. The last three sections show how **data leakage** breaks all of these tools, when information from the validation rows, or information that will not exist at prediction time, slips into training, and how a scikit-learn `Pipeline` prevents the most common form.

The code cells build on each other; run them in order. The first cell moves to the repository root, where the case-study paths start. The Telco churn data (7,043 customers) are downloaded from IBM's GitHub repository and serve as a small warm-up. The main example is the Inside Airbnb listings for Berlin in `case-study/data/airbnb/` (`uv run python case-study/prepare_airbnb.py`, see [case-study/README.md](../../../case-study/README.md)).

```mermaid
flowchart LR
    A["All labelled data"] --> B["Development data"]
    A --> T["Test set<br/>(locked away)"]
    B --> CV["Cross-validation:<br/>train and validate k times,<br/>preprocessing inside a Pipeline"]
    CV --> C["Choose model and<br/>hyperparameters"]
    C --> F["Refit on all<br/>development data"]
    F --> E["Evaluate once<br/>on the test set"]
    T --> E
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Training, validation and test data and their roles

**Concept.** Labelled data are used for three different jobs, and each job needs its own rows.

- The **training set** is used to fit the model: the algorithm sees these rows and their labels and chooses the parameters (for example the coefficients of a regression).
- The **validation set** is used to make choices: which model, which features, which settings. We compare candidates by their validation score.
- The **test set** is used once, at the end, to estimate how well the chosen model will perform on new data. It must not influence any choice.

A worked example. A bank has 10,000 past loan applications with known outcomes. It keeps 2,000 applications aside as the test set. Of the remaining 8,000 it uses 6,000 to fit three candidate models and 2,000 to compare them. Model B wins on the validation rows. Only now does the bank score model B on the 2,000 test applications: 81 % accuracy. This number is an honest estimate because no decision depended on the test rows. The validation score of model B (say 83 %) is slightly optimistic: B was chosen *because* it did well on those rows.

**Why it matters.** Each time a score is used to make a decision, the rows behind it stop being "new data". If you choose a model by its test score, the test score is no longer an unbiased estimate; with enough candidates one of them will look good by chance. Keeping the three roles apart is the basic discipline of all model evaluation. In this course the leaderboard of Sessions 13–16 plays the role of the test set: its labels are hidden, so nobody can tune on them.

**How it works in Python.** Two calls of `train_test_split` give a 60/20/20 split. `stratify=y` keeps the share of churners equal in all three parts.

In [2]:
import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(URL)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
y = (df["Churn"] == "Yes").astype(int)               # 1 = customer left
X = df.drop(columns=["customerID", "Churn"])

# 20 % test set, then 25 % of the remaining 80 % as validation set
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
X_tr, X_val, y_tr, y_val = train_test_split(X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=0)
print(len(X_tr), len(X_val), len(X_test))           # 4225 1409 1409

num = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
cat = [c for c in X.columns if c not in num]
prep = make_column_transformer((StandardScaler(), num), (OneHotEncoder(handle_unknown="ignore"), cat))

# choose the regularisation strength C on the validation set (more on C in lecture notebook 05)
for C in [0.001, 0.01, 1.0]:
    m = make_pipeline(prep, LogisticRegression(C=C, max_iter=1000)).fit(X_tr, y_tr)
    print(C, round(m.score(X_tr, y_tr), 3), round(m.score(X_val, y_val), 3))
# 0.001 0.773 0.767
# 0.01  0.806 0.796
# 1.0   0.807 0.803   <- best validation accuracy

# refit the chosen setting on training + validation data, evaluate ONCE on the test set
final = make_pipeline(prep, LogisticRegression(C=1.0, max_iter=1000)).fit(X_dev, y_dev)
print(round(final.score(X_test, y_test), 3))        # 0.801

4225 1409 1409


0.001 0.773 0.767
0.01 0.806 0.796


1.0 0.807 0.803
0.801


**In practice.**
- The Netflix Prize (2006–2009) kept a "quiz" set for the public leaderboard and a separate hidden "test" set that decided the winner, so that teams could not win by tuning to the leaderboard.
- The TRIPOD reporting guideline for clinical prediction models (Collins et al., 2015) asks authors to state how data were split for development and validation, and treats a separate external test as stronger evidence than internal validation.
- Kaggle competitions show a public leaderboard during the competition and rank by a private part of the test set at the end. Rankings often move between the two ("shake-up"), which shows how much teams had adapted to the public part.

> **Warning.** Looking at the test score "just to see" and then changing the model turns the test set into a validation set. Decide on the model first, then look at the test score once and report it, even if it is disappointing.

> **Tip.** Fix `random_state` in every split and every model that uses randomness. Then a colleague who runs your notebook gets the same numbers.

## Why a single split is unreliable

**Concept.** A test score is computed on a finite, randomly chosen set of rows. Another random split puts other customers into the test set and gives another score. The score is therefore a **random variable**: it has a mean (the quantity we want, the expected performance on new data) and a spread (noise caused by the particular split). The spread shrinks with the size of the test set: for accuracy, the standard error is roughly √(p(1 − p)/n). With p = 0.8 and n = 1,761 test customers this is √(0.16/1761) ≈ 0.0095, about one percentage point. With n = 200 it is almost three points.

**Why it matters.** Model comparisons, hyperparameter choices and project reports all rest on such estimates. If the score moves by two points when only the seed changes, an "improvement" of one point is not evidence. A single split also wastes data: the rows in the test set never help to train, and the rows in the training set never help to evaluate.

**How it works in Python.** The same model, evaluated on 20 random 75/25 splits:

In [3]:
model = make_pipeline(prep, LogisticRegression(max_iter=1000))

acc = []
for seed in range(20):
    a_tr, a_te, b_tr, b_te = train_test_split(X, y, test_size=0.25, random_state=seed)
    acc.append(model.fit(a_tr, b_tr).score(a_te, b_te))
print(np.round([min(acc), np.mean(acc), max(acc)], 3))   # [0.785 0.802 0.822]
print(round(np.std(acc), 3))                             # 0.009

[0.785 0.802 0.822]
0.009


The range from 0.785 to 0.822 comes only from chance. The observed standard deviation (0.009) agrees with the formula above.

**In practice.**
- Kaggle "shake-ups": in competitions with small test sets, teams that ranked high on the public leaderboard regularly drop many places on the private leaderboard, because the public part was too small to separate models.
- The ImageNet replication study by Recht et al. (2019) built new test sets for CIFAR-10 and ImageNet and found accuracy drops of 3–15 percentage points on the new samples. The authors attribute the drops mainly to small differences in how the new images were collected, not to adaptive overfitting: a test score is tied to the particular test sample.

> **Caution.** Never report a difference between two models that is smaller than the spread of the score across splits as an improvement. Report the spread (or an interval) along with the score.

> 📖 **Read more:** [extras/02-train-test-and-cross-validation.ipynb](extras/02-train-test-and-cross-validation.ipynb) (INRIA scikit-learn MOOC): one train–test split against many shuffled splits on the California housing data, with the spread of the test error made visible.

## k-fold and stratified cross-validation

**Concept.** **k-fold cross-validation** divides the rows into *k* parts (**folds**) of roughly equal size. The model is trained *k* times, each time on *k* − 1 folds, and scored on the remaining fold. Every row is used for validation exactly once. The *k* scores are summarised by their mean (the estimate) and their standard deviation (how stable it is). Common choices are *k* = 5 or *k* = 10; **leave-one-out** (*k* = *n*) is the extreme case.

![Five-fold cross-validation (left): each row of blocks is one split; the orange block is the validation fold, the blue blocks are used for training. Time-series split (right): training always lies before validation.](../figures/cv_splits.png)

A small example by hand: 10 rows, *k* = 5. Fold 1 = rows 1–2, fold 2 = rows 3–4, and so on. In split 1 the model trains on rows 3–10 and is scored on rows 1–2; in split 2 it trains on rows 1–2 and 5–10 and is scored on rows 3–4. After five splits each row has been predicted once by a model that did not see it.

**Stratified k-fold** chooses the folds so that each has the same class proportions as the full data. With 26.5 % churners, plain k-fold can produce folds with 24.6 % and 28.9 % churn; stratified folds all have 26.5 %. This matters most for rare classes and small data.

**Why it matters.** Cross-validation uses all data for both training and validation, so the estimate has lower variance than one split, and the fold-to-fold spread shows how much to trust it. Comparing training and validation scores in the same run shows over- or underfitting (lecture notebook 05).

**How it works in Python.** `cross_val_score` returns one score per fold; `cross_validate` returns several metrics, fit times and, optionally, training scores. For classifiers, `cv=5` already means stratified 5-fold; pass a splitter object to control shuffling and the seed.

In [4]:
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score, cross_validate

kf = KFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(model, X, y, cv=kf, scoring="accuracy")
print(scores.round(3))                                    # [0.797 0.813 0.797 0.796 0.808]
print(round(scores.mean(), 3), round(scores.std(), 3))    # 0.802 0.007

# share of churners in each validation fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
print(np.round([y.iloc[va].mean() for _, va in kf.split(X, y)], 3))    # [0.261 0.269 0.289 0.262 0.246]
print(np.round([y.iloc[va].mean() for _, va in skf.split(X, y)], 3))   # [0.265 0.265 0.265 0.265 0.266]

# several metrics at once, with training scores
res = cross_validate(model, X, y, cv=skf, scoring=["accuracy", "roc_auc", "f1"], return_train_score=True)
for m in ["accuracy", "roc_auc", "f1"]:
    print(m, round(res[f"train_{m}"].mean(), 3), round(res[f"test_{m}"].mean(), 3))
# accuracy 0.807 0.803
# roc_auc  0.849 0.845
# f1       0.603 0.596   (training and validation close: no sign of overfitting)

[0.797 0.813 0.797 0.796 0.808]
0.802 0.007
[0.261 0.269 0.289 0.262 0.246]
[0.265 0.265 0.265 0.265 0.266]


accuracy 0.807 0.803
roc_auc 0.849 0.845
f1 0.603 0.596


> **Note.** In scikit-learn the validation scores of `cross_validate` are called `test_...`. They are validation scores in the sense of this notebook: they come from the development data, not from a locked test set.

**In practice.**
- scikit-learn uses 5-fold cross-validation (stratified for classifiers) as the default in `cross_val_score`, `GridSearchCV` and related tools.
- In genomics and medical imaging, where studies often have only a few hundred patients, k-fold cross-validation is the usual way to report performance; ISLP (James et al., 2023, Ch. 5) recommends *k* = 5 or 10 as a good bias–variance compromise.
- Credit scoring and churn teams typically compare candidate models by cross-validated ROC AUC before a final out-of-time test.

> **Warning.** The *k* models of cross-validation are thrown away. Cross-validation estimates the performance of a *procedure* (this model type with these settings). To use the model, refit it on all development data.

> **Caution.** Use `shuffle=True` in `KFold` when the rows are sorted (for example by date or by class). Without shuffling, the first fold contains only the oldest rows or only one class. If rows are sorted *because time matters*, do not shuffle: use a time-series split (next section).

> 📖 **Read more:** [extras/01-model-validation.ipynb](extras/01-model-validation.ipynb) (Python Data Science Handbook, ch. 5.3): shows on the iris data why a 1-nearest-neighbour model scores 100 % on its own training data, then compares a holdout set, two-fold and leave-one-out cross-validation.
>
> 📖 **Read more:** [extras/03-stratified-cross-validation.ipynb](extras/03-stratified-cross-validation.ipynb) (INRIA scikit-learn MOOC): on the iris data sorted by class, unshuffled `KFold` scores 0 % because every test fold holds a class the model never saw; `StratifiedKFold` keeps the class shares in every fold.
>
> 📖 **Read more:** [extras/05-islp-resampling-lab.ipynb](extras/05-islp-resampling-lab.ipynb) (ISLP labs): ISLP Lab 5 compares the validation-set approach, leave-one-out and k-fold cross-validation for choosing a polynomial degree, and computes bootstrap standard errors of regression coefficients.

## Grouped and time-series splits

**Concept.** k-fold assumes that rows are independent and that the future looks like a random sample of the past. Two common situations break this.

1. **Groups.** Several rows can belong to the same unit: several visits of the same patient, several transactions of the same customer, several Airbnb listings of the same host. If a group appears in both training and validation folds, the model can recognise the group instead of learning a general pattern. **`GroupKFold`** (or `GroupShuffleSplit` for a single split) assigns whole groups to folds, so no group appears on both sides. It needs a `groups` array, here the host identifier.
2. **Time.** If the model will predict the future, a random split lets it train on rows recorded *after* the ones it is validated on. This hides changes over time (the COVID break in Berlin's Airbnb market, a new registration rule, new kinds of offers), which are called **drift**. **`TimeSeriesSplit`** needs rows sorted by time. Split *i* trains on the first part and validates on the block that follows; the training window grows with each split (right panel of the figure above). A single **out-of-time** split (train on the older years, validate on the most recent one) is the simplest version. Session 12 uses this scheme to backtest the demand forecast, and Session 13 compares random and time-based validation for the customs text classifier of the leaderboard.

```mermaid
flowchart TD
    Q1{"Will the model be used<br/>on new groups?<br/>(new hosts, patients)"} -->|yes| G["GroupKFold"]
    Q1 -->|no| Q2{"Will it predict<br/>the future?"}
    Q2 -->|yes| TS["TimeSeriesSplit or<br/>one out-of-time split"]
    Q2 -->|no| Q3{"Classification?"}
    Q3 -->|yes| S["StratifiedKFold"]
    Q3 -->|no| K["KFold (shuffle=True)"]
```

**Why it matters.** The splitter must reproduce the situation in which the model will be used. Ordinary k-fold measures performance on groups and periods already seen; the real use often involves new ones: a price suggestion is most useful for a host who has never listed before, and a forecast is always about next month.

**How it works in Python: groups.** A price model for Berlin Airbnb listings: predict the log of the nightly price from size, location, room type and reviews with gradient boosting (Session 10; here a black box). The table is the short-stay table of Sessions 4–6 (6,701 listings with a price and a minimum stay below 28 nights) without the 26 listings priced below €10 or above €1,000, which Session 4 treats as outliers. Listings with a minimum stay of 28 nights or more are medium-term rentals with a different price basis (median €23 a night in this snapshot) and belong to another model. Many hosts offer several similar flats, often in the same building, with similar prices.

In [5]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold

lst = pd.read_parquet("case-study/data/airbnb/listings.parquet")
bnb = lst[(lst["minimum_nights"] < 28) & lst["price"].between(10, 1000)].reset_index(drop=True)
bnb["dist_km"] = np.hypot((bnb["latitude"] - 52.5219) * 111.2,        # km to Alexanderplatz
                          (bnb["longitude"] - 13.4132) * 68.0)
cols = ["accommodates", "bedrooms", "beds", "bathrooms", "dist_km", "room_type", "district",
        "number_of_reviews", "review_scores_rating", "availability_365"]
X_bnb = pd.get_dummies(bnb[cols], columns=["room_type", "district"], dtype=int)
y_bnb = np.log(bnb["price"])                                         # log of the nightly price in EUR
hosts = bnb["host_id"]
n_per_host = hosts.map(hosts.value_counts())
print(len(bnb), hosts.nunique(), round((n_per_host > 1).mean(), 3), n_per_host.max())
# 6675 3737 0.565 83   <- listings, hosts, share of listings whose host has several, largest host

gbm = HistGradientBoostingRegressor(random_state=0)
random_cv = cross_val_score(gbm, X_bnb, y_bnb, cv=KFold(5, shuffle=True, random_state=0), scoring="r2")
grouped_cv = cross_val_score(gbm, X_bnb, y_bnb, cv=GroupKFold(5), groups=hosts, scoring="r2")
print(random_cv.mean().round(3), random_cv.std().round(3))       # 0.636 0.033
print(grouped_cv.mean().round(3), grouped_cv.std().round(3))     # 0.583 0.022

6675 3737 0.565 83


0.636 0.033
0.583 0.022


Random 5-fold cross-validation reports R² = 0.64; folds that keep each host on one side report 0.58. The difference is larger than the fold-to-fold spread, so it is not noise. In a random split, 55 % of the validation listings have a host whose other listings sit in the training folds, and for hosts with several listings the host's average alone explains 70 % of the variance of the log price. The model partly recognises "a flat of this host" instead of learning what size and location are worth. The effect depends on the model: the small linear model of Session 6 (guests, room type, distance, district) scores 0.513 with random and 0.502 with grouped folds, because four features leave little room to memorise hosts; the more flexible a model and the richer its features, the larger the gap (lecture notebook 05 and the worked example 04). Which number is right depends on the use. A price suggestion for a **new host** should be judged with `GroupKFold`; a tool that only prices further flats of hosts already in the data may use the random estimate. The [Airbnb validation workbook](04-case-study-airbnb-price-validation.ipynb) repeats the comparison for a linear model and a random forest.

> **Note.** Inside Airbnb collects these data from public listing pages (CC BY 4.0; snapshot of 26 June 2026). The course copy has no host names. Use `host_id` only as a grouping key and report results in aggregate; never look up or name individual hosts.

**How it works in Python: time.** `TimeSeriesSplit(5)` on rows sorted by date returns five splits in which every training index lies before every validation index. The listings snapshot is a single day and has no time order to respect, so this notebook does not use it; the worked example [02-cross-validation-splitters.ipynb](02-cross-validation-splitters.ipynb) shows the indices, and Session 12 applies the scheme to monthly demand.

**In practice.**
- Medical imaging: the CheXNet study (Rajpurkar et al., 2017) split the ChestX-ray14 images by patient so that no patient appeared in both training and test data; otherwise a model can recognise the patient rather than the disease.
- Credit risk models in banks are routinely validated out-of-time: trained on older loans and tested on more recent ones.
- Demand and energy-load forecasting use rolling-origin evaluation, the forecasting name for the same scheme (Session 12).

> **Warning.** `TimeSeriesSplit` uses the row order. Sort by date first (`sort_values("date", ignore_index=True)`); otherwise the "past" and "future" are arbitrary.

> **Caution.** Grouping and time can both apply. A model for next year's prices of *new* hosts would need both: hosts kept together and the latest period held out. Decide which situation matters for your project and say so in the validation plan.

> 📖 **Read more:** [extras/04-grouped-cross-validation.ipynb](extras/04-grouped-cross-validation.ipynb) (INRIA scikit-learn MOOC): the digits data contain many samples by the same writers; shuffled k-fold overestimates the accuracy, and `GroupKFold` by writer gives the lower, honest estimate.
>
> 📖 **Read more:** [extras/07-honest-validation-practice.ipynb](extras/07-honest-validation-practice.ipynb) (course notebook): six synthetic listings, small enough to follow by hand, show that a random split and an unseen-host split answer different questions (MAE €10 against €120).

## Bootstrap confidence interval of a metric

**Concept.** The **bootstrap** (Efron, 1979; recap in Session 5) estimates the uncertainty of a statistic from one sample: draw *n* rows **with replacement** from the *n* rows, recompute the statistic, and repeat about 1,000 times. The spread of the recomputed values approximates the sampling variability. For a model, the predictions stay fixed and only the test rows are resampled. The 2.5 % and 97.5 % percentiles of the bootstrap values form a 95 % **percentile confidence interval** for the test metric.

A tiny example by hand. Five test listings with absolute price errors of €10, €20, €30, €40 and €100: the mean absolute error (MAE) is €40. One bootstrap sample draws positions 2, 2, 3, 5, 1 → €20, €20, €30, €100, €10 → MAE €36; another draws 5, 5, 4, 1, 5 → €100, €100, €40, €10, €100 → €70. Repeating this many times gives a distribution of MAEs; with only five rows it is very wide, which is the honest answer.

**Why it matters.** A score without an interval cannot tell whether a difference between two models, or between a validation score and a later test score, is larger than chance. The bootstrap works for any metric, including MAE in euros, macro-F1 and ROC AUC, for which no simple formula exists.

**How it works in Python.** The question of the practice: how far off would the price model be for a host who lists for the first time? Lock away 20 % of the hosts with all their listings, fit the gradient-boosting model on the other hosts, and compute the MAE in euros on the locked hosts. Then bootstrap it twice: once resampling listings, once resampling **hosts** (a **cluster bootstrap**, because listings of the same host are not independent).

In [6]:
from sklearn.model_selection import GroupShuffleSplit

dev, test = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0).split(X_bnb, y_bnb, groups=hosts))
gbm.fit(X_bnb.iloc[dev], y_bnb.iloc[dev])                      # trained on 80 % of the hosts
price = bnb["price"].iloc[test].to_numpy()
abs_err = np.abs(price - np.exp(gbm.predict(X_bnb.iloc[test])))  # error in euros per listing
print(len(test), hosts.iloc[test].nunique(), round(abs_err.mean(), 1), np.median(price))
# 1254 748 50.8 160.0   <- test listings, test hosts, MAE in EUR, median price of the test listings

rng = np.random.default_rng(0)
boot_rows = [abs_err[rng.integers(0, len(abs_err), len(abs_err))].mean() for _ in range(1000)]
by_host = pd.DataFrame({"host": hosts.iloc[test].to_numpy(), "err": abs_err}).groupby("host")["err"]
sums, counts = by_host.sum().to_numpy(), by_host.count().to_numpy()
boot_hosts = []
for _ in range(1000):                                           # draw hosts with replacement
    pick = rng.integers(0, len(sums), len(sums))
    boot_hosts.append(sums[pick].sum() / counts[pick].sum())
print(np.percentile(boot_rows, [2.5, 97.5]).round(1))           # [47.4 54.8]  listings resampled
print(np.percentile(boot_hosts, [2.5, 97.5]).round(1))          # [45.5 56.5]  hosts resampled

1254 748 50.8 160.0
[47.4 54.8]
[45.5 56.5]


For a new host the model is off by about €51 a night on average, with a 95 % interval of roughly €46 to €57: about a third of the median price of €160. The interval that resamples listings is narrower (±€3.7 against ±€5.5) because it treats 1,254 listings as independent although they come from only 748 hosts, and a host's listings tend to be all well or all badly priced by the model. A model that reaches €49 instead of €51 on this test set is not demonstrably better.

> **Note.** Two related summaries are easy to confuse. The **standard deviation across CV folds** describes how much the score varies between training sets and validation folds. The **bootstrap interval on a test set** describes the uncertainty from the finite test sample for one fitted model. Both are useful; say which one you report.

**In practice.**
- Diagnostic accuracy studies in medicine report sensitivity, specificity and AUC with 95 % confidence intervals; bootstrap intervals are common for AUC.
- Machine translation research uses paired bootstrap resampling (Koehn, 2004) to test whether one system's BLEU score is better than another's on the same test set.

> **Warning.** Resample the *test rows*, not the training rows, when you want the uncertainty of a test score. Refitting the model inside every bootstrap round answers a different question (the variability of the training procedure) and is far slower.

> **Caution.** The bootstrap assumes that the test rows are independent. If they come in groups (listings of one host, visits of one patient, purchases of one customer), resample whole groups as above, otherwise the interval is too narrow.

> 📖 **Read more:** [extras/06-bootstrap-and-simulation.ipynb](extras/06-bootstrap-and-simulation.ipynb) (statsthinking21): simulates a sampling distribution (the maximum of finishing times) and checks that the bootstrap distribution of the mean matches the theoretical one.
>
> 📖 **Read more:** [extras/16-case-study-churn-validation.ipynb](extras/16-case-study-churn-validation.ipynb) (course notebook): part A reports the cross-validated ROC AUC of the Telco churn model and a bootstrap 95 % interval on its test set.

## Data leakage: preprocessing outside the cross-validation

**Concept.** Many preparation steps *learn* something from the data: a `StandardScaler` learns means and standard deviations, a `SimpleImputer` learns medians, a feature selector learns which columns are related to the target, a `CountVectorizer` learns a vocabulary. If such a step is fitted on **all** rows before the data are split, the validation folds have influenced the training. The model is then evaluated on rows that were not truly unseen.

How much this matters depends on the step. Scaling with the overall mean instead of the training mean changes little when the data are large. A step that looks at the **target** (feature selection, target encoding, oversampling of rare classes in Session 8) can change the result completely, as the example below shows.

Worked example: 200 rows, 5,000 columns of pure noise, random labels. No model can do better than 50 % accuracy. But among 5,000 random columns, some will by chance be correlated with the labels *in these 200 rows*. If we select the 20 most correlated columns using all rows, and then cross-validate, the model finds those chance correlations again in every validation fold, because the validation rows helped to pick them.

**Why it matters.** Leakage is one of the most common reasons why published or deployed models perform worse than reported. Kapoor and Narayanan (2023) reviewed studies in 17 scientific fields and found leakage in hundreds of papers, often leading to strongly over-optimistic claims.

**How it works in Python.** First the extreme case on pure noise, then the Berlin price model of the earlier sections.

In [7]:
import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold, KFold, cross_val_predict, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder

# pure noise: 200 rows, 5,000 random features, random labels -> true accuracy is 0.5
rng = np.random.default_rng(0)
X_noise, y_noise = rng.normal(size=(200, 5000)), rng.integers(0, 2, 200)

# WRONG: pick the 20 features most related to y using ALL rows, then cross-validate
X_sel = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
print(cross_val_score(LogisticRegression(), X_sel, y_noise, cv=5).mean().round(2))      # 0.8

# RIGHT: the selector is part of the pipeline and is refitted inside every training fold
leak_free = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())
print(cross_val_score(leak_free, X_noise, y_noise, cv=5).mean().round(2))               # 0.51

0.8
0.51


The Berlin price model: 6,675 short-stay listings priced between €10 and €1,000 (section on grouped splits above), the log of the nightly price as target, size, distance to Alexanderplatz, reviews and availability as numbers, room type, district and property type as categories, and the 138 neighbourhoods (Ortsteile). The helper reports R² on the log scale and the mean absolute error in euros, both from cross-validated predictions.

In [8]:
lst = pd.read_parquet("case-study/data/airbnb/listings.parquet")
bnb = lst[(lst["minimum_nights"] < 28) & lst["price"].between(10, 1000)].reset_index(drop=True)
bnb["dist_km"] = np.hypot((bnb["latitude"] - 52.5219) * 111.2, (bnb["longitude"] - 13.4132) * 68.0)
num = ["accommodates", "bedrooms", "beds", "bathrooms", "dist_km", "minimum_nights",
       "availability_365", "number_of_reviews", "review_scores_rating"]
cat = ["room_type", "district", "property_type"]
y = np.log(bnb["price"])
hosts = bnb["host_id"]
X = bnb[num + cat + ["neighbourhood"]].assign(host=hosts.astype(str))
random5, grouped5 = KFold(5, shuffle=True, random_state=0), GroupKFold(5)
num_prep = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler())

def cv_scores(model, X, cv, groups=None):
    pred = cross_val_predict(model, X, y, cv=cv, groups=groups)
    return f"R² {r2_score(y, pred):.3f}  MAE € {mean_absolute_error(bnb['price'], np.exp(pred)):.1f}"

# (a) imputer and scaler fitted on ALL rows, then cross-validated
X_scaled = pd.DataFrame(num_prep.fit_transform(X[num])).add_prefix("z").join(X[cat + ["neighbourhood"]])
one_hot = make_column_transformer((OneHotEncoder(handle_unknown="ignore"), cat + ["neighbourhood"]),
                                  remainder="passthrough")
print(cv_scores(make_pipeline(one_hot, Ridge(alpha=10)), X_scaled, random5))     # R² 0.611  MAE € 50.6
inside = make_column_transformer((num_prep, num), (OneHotEncoder(handle_unknown="ignore"), cat + ["neighbourhood"]))
print(cv_scores(make_pipeline(inside, Ridge(alpha=10)), X, random5))             # R² 0.611  MAE € 50.6

# (b) a hand-made target encoding: mean log price of the neighbourhood, from ALL rows
X_nb = X.assign(nb_price=y.groupby(X["neighbourhood"]).transform("mean"))
leaky_nb = make_column_transformer((num_prep, num + ["nb_price"]), (OneHotEncoder(handle_unknown="ignore"), cat))
print(cv_scores(make_pipeline(leaky_nb, Ridge(alpha=10)), X_nb, random5))       # R² 0.614  MAE € 50.6
# the same encoding learned inside the pipeline, from the training folds only
nb_inside = make_column_transformer((num_prep, num), (OneHotEncoder(handle_unknown="ignore"), cat),
                                    (TargetEncoder(cv=KFold(5, shuffle=True, random_state=0)), ["neighbourhood"]))
print(cv_scores(make_pipeline(nb_inside, Ridge(alpha=10)), X, random5))          # R² 0.605  MAE € 51.0

R² 0.611  MAE € 50.6


R² 0.611  MAE € 50.6


R² 0.614  MAE € 50.6


R² 0.605  MAE € 51.0


Scaling with all rows changes nothing visible: means and standard deviations of 6,675 listings hardly move when one fold is added. The neighbourhood means are a step that looks at the target: fitted on all rows, every listing's own price has gone into the mean of its neighbourhood. With about 50 listings per neighbourhood the effect is still small here (0.614 against 0.605 for the same encoding learned inside the pipeline; `TargetEncoder` is explained in the next section and in Session 9). The next section shows the same mistake where the groups are small.

**In practice.**
- Kapoor and Narayanan (2023) traced a series of over-optimistic results in civil-war prediction to leakage, including imputation fitted on the full dataset; after correction, the complex models did not beat a logistic regression.
- The scikit-learn documentation has a page "Common pitfalls and recommended practices" whose first topic is preprocessing outside the cross-validation, with the same feature-selection example. The worked example [03-data-leakage-feature-selection.ipynb](03-data-leakage-feature-selection.ipynb) (INRIA scikit-learn MOOC) walks through it step by step.

> **Caution.** Any step that looks at the target (feature selection by correlation with y, target encoding, resampling such as SMOTE in Session 8) must be fitted inside the cross-validation. Fitted on all rows, it produces scores that cannot be trusted at all.

## Data leakage: target leakage

**Concept.** **Target leakage** means that a feature contains information about the target that will not be available when the model is used. Typical sources:

- a feature *computed from the target*, for example a customer's average rating that includes the rating being predicted;
- a feature *recorded after the event*, for example "reason for cancellation" in a churn table, or "antibiotic prescribed" when predicting an infection;
- an identifier or timestamp that happens to correlate with the label in the collected data.

The model learns the shortcut, cross-validation confirms it (the shortcut is present in every fold), and the model fails in use, where the shortcut is missing.

**Why it matters.** No splitter can detect target leakage: the leak is inside the rows. It can only be found by asking, for every feature, *"would I know this value at the moment of prediction?"* A tempting feature for the price model is the **host's average price**: hosts with expensive flats tend to have more expensive flats. Computed from all listings, it contains the listing's own price, and 43.5 % of the listings belong to hosts with a single listing, for whom the "host average" *is* the target. A first-time host has no average at all.

**How it works in Python.**

In [9]:
# the host's mean log price from ALL listings, including the listing being predicted
X_host = X_nb.assign(host_price=y.groupby(X["host"]).transform("mean"))
leaky_host = make_column_transformer((num_prep, num + ["nb_price", "host_price"]),
                                     (OneHotEncoder(handle_unknown="ignore"), cat))
print(cv_scores(make_pipeline(leaky_host, Ridge(alpha=10)), X_host, random5))           # R² 0.869  MAE € 27.7
print(cv_scores(make_pipeline(leaky_host, Ridge(alpha=10)), X_host, grouped5, hosts))   # R² 0.866  MAE € 27.8

R² 0.869  MAE € 27.7


R² 0.866  MAE € 27.8


Cross-validation now reports an error of €28 instead of €51, and host-grouped folds do not help (0.866): the leak sits inside each row, so it travels with the row into whichever fold it lands in. For a new host the column would be empty. The leak would have been found by the question above, or by its symptom: one new feature that halves the error is a warning sign. Session 9 meets a second kind, the revenue estimate of Inside Airbnb, which is computed from the price and the occupancy that a demand model is meant to predict.

**In practice.**
- KDD Cup 2008 (breast cancer detection): the patient identifier was predictive of the label because of how the data had been assembled; Kaufman et al. (2012) use it as a textbook case of leakage.
- Hospital data: a feature such as "antibiotic prescribed" or "chest X-ray ordered" can reveal the diagnosis a model is supposed to predict, because it is recorded after the doctor suspected it.
- Churn data: fields like "reason for leaving" or "contract end date" are filled only for customers who have already left.
- Marketplace data: an average price, rating or revenue per seller computed from all rows contains the row being predicted, and it does not exist for a new seller.

> **Warning.** A suspiciously large jump in validation score from one new feature is a warning sign, not a success. Check when and how that feature is recorded before you celebrate.

## Preprocessing inside a Pipeline

**Concept.** A scikit-learn **`Pipeline`** chains steps: zero or more transformers (imputer, scaler, encoder, selector, vectoriser) and a final model. Calling `fit` on the pipeline fits each step on the training data and passes the transformed data on; calling `predict` or `score` only *applies* the fitted steps to new data. When a pipeline is passed to `cross_val_score`, `GridSearchCV` or `cross_validate`, the whole pipeline is refitted on the training folds of every split, so the validation fold never influences any step.

```mermaid
sequenceDiagram
    participant CV as cross_val_score
    participant P as Pipeline
    participant Tr as Training folds
    participant Va as Validation fold
    CV->>P: fit(training folds)
    P->>Tr: imputer and scaler learn medians, means
    P->>Tr: model learns coefficients
    CV->>P: score(validation fold)
    P->>Va: apply stored medians, means
    P->>Va: predict and score
```

**Why it matters.** The pipeline turns "remember to fit the scaler only on the training part" from a rule that people forget into code that cannot get it wrong. It also packages preparation and model into one object that can be saved and deployed (Session 16), so the same steps are applied to new data in production. Session 8 extends it with `ColumnTransformer` for mixed numeric and categorical columns.

**How it works in Python.** The fix for both columns above is `TargetEncoder` *inside* the pipeline: in every split it learns the neighbourhood and host means from the training folds only, and it cross-fits them within the training folds as well (Session 9 explains how). Steps are named automatically by `make_column_transformer` and `make_pipeline` (lower-case class names), which is how a search addresses their hyperparameters.

In [10]:
encoded = make_column_transformer((num_prep, num), (OneHotEncoder(handle_unknown="ignore"), cat),
                                  (TargetEncoder(cv=KFold(5, shuffle=True, random_state=0)), ["neighbourhood", "host"]))
pipe = make_pipeline(encoded, Ridge(alpha=10))
print(list(pipe.named_steps))                                   # ['columntransformer', 'ridge']
print(cv_scores(pipe, X, random5))                              # R² 0.645  MAE € 48.0

['columntransformer', 'ridge']


R² 0.645  MAE € 48.0


The error rises from €28 to €48: most of the leak is gone. But not all of it. With random folds, 55 % of the validation listings have a host whose other listings sit in the training folds, so the encoder still knows that host's prices. That is the **second leaking workflow: the same host in training and validation**. A first-time host is never in the training data, and only grouped folds reproduce that:

In [11]:
print(cv_scores(pipe, X, grouped5, hosts))                      # R² 0.574  MAE € 52.1

no_host = make_column_transformer((num_prep, num), (OneHotEncoder(handle_unknown="ignore"), cat),
                                  (TargetEncoder(cv=KFold(5, shuffle=True, random_state=0)), ["neighbourhood"]))
print(cv_scores(make_pipeline(no_host, Ridge(alpha=10)), X, grouped5, hosts))   # R² 0.587  MAE € 51.9

R² 0.574  MAE € 52.1


R² 0.587  MAE € 51.9


| Workflow | Cross-validated R² | MAE € |
|---|---|---|
| Host and neighbourhood means from all rows, random folds | 0.869 | 27.7 |
| The same, host-grouped folds | 0.866 | 27.8 |
| Target encoding inside the Pipeline, random folds | 0.645 | 48.0 |
| Target encoding inside the Pipeline, host-grouped folds | 0.574 | 52.1 |
| Without the host column, host-grouped folds | 0.587 | 51.9 |

Each leak needs its own fix: the pipeline removes the target from the features, the grouped folds remove the host's other listings from training. Once both are fixed, the host column is worth nothing for a new host (an unknown host gets the overall mean), and the simpler model without it is slightly better. The worked example [04-case-study-airbnb-price-validation.ipynb](04-case-study-airbnb-price-validation.ipynb), part C, repeats the comparison with gradient boosting.

> **Tip.** If a step needs the target or learns anything from the data (medians, means, categories, a vocabulary), it belongs in the pipeline. If it is a fixed rule that looks at one row at a time (for example the distance to Alexanderplatz from the coordinates, or `np.log` of the price), it can be applied before splitting.

**In practice.**
- Production machine learning systems serialise the fitted pipeline (for example with `joblib` or `skops`) rather than the model alone, so that the serving code cannot apply different preprocessing; this "training–serving skew" is a known source of failures described in Google's *Rules of Machine Learning* (Zinkevich).
- The INRIA scikit-learn MOOC uses pipelines from its first module onwards for exactly this reason.

> **Warning.** Manual `fit_transform` on the training data and `transform` on the test data is correct for one split, but breaks as soon as you cross-validate or tune: you would have to repeat it by hand for every fold. Use a pipeline.

## Check your understanding

1. You tried 30 models and picked the one with the best validation score. Why is that validation score an optimistic estimate, and what number should you report instead?
2. For each case, choose a splitter and justify it: (a) predicting next month's churn; (b) classifying 500 tumour images from 120 patients; (c) a price suggestion for people who list their first flat on Airbnb.
3. Describe in three steps how to compute a 95 % cluster-bootstrap interval for the MAE of the price model on held-out hosts.
4. Why does fitting a `StandardScaler` on all rows usually matter little, while fitting `SelectKBest` on all rows can matter a lot?
5. Give one example of target leakage in a churn table and one in the Airbnb listings. How would you detect them?

## Further reading

- James, G., Witten, D., Hastie, T., Tibshirani, R. & Taylor, J. (2023). *An Introduction to Statistical Learning with Applications in Python*, Chapter 5 "Resampling Methods". Springer. https://www.statlearning.com/
- scikit-learn developers (2025). *Cross-validation: evaluating estimator performance* and *Common pitfalls and recommended practices*. https://scikit-learn.org/stable/modules/cross_validation.html · https://scikit-learn.org/stable/common_pitfalls.html
- Kapoor, S. & Narayanan, A. (2023). Leakage and the reproducibility crisis in machine-learning-based science. *Patterns*, 4(9), 100804. https://doi.org/10.1016/j.patter.2023.100804
- Kaufman, S., Rosset, S., Perlich, C. & Stitelman, O. (2012). Leakage in data mining: formulation, detection, and avoidance. *ACM Transactions on Knowledge Discovery from Data*, 6(4), 15. https://doi.org/10.1145/2382577.2382579
- Raschka, S. (2018). *Model Evaluation, Model Selection, and Algorithm Selection in Machine Learning*. arXiv:1811.12808. https://arxiv.org/abs/1811.12808